# Chapter 21: Asynchronous Programming
*Fluent Python, 2nd Edition -- Luciano Ramalho*

This notebook distills the key ideas from Chapter 21 into runnable, self-contained examples.
All code targets **Python 3.11+** and runs entirely in standard Jupyter kernels using `asyncio.run()` or `nest_asyncio`.

**Concepts covered:**
1. Native coroutines and `async`/`await` basics
2. The asyncio event loop (`asyncio.run`, `gather`, `as_completed`)
3. Async iterators and async generators
4. Async context managers (`async with`, `@asynccontextmanager`)
5. Semaphores for throttling concurrent tasks
6. Async comprehensions
7. How async works -- and when it does not

In [ ]:
# Setup: enable asyncio.run() inside Jupyter notebooks
# (Jupyter already runs an event loop, so we patch it)
import asyncio
try:
    import nest_asyncio
    nest_asyncio.apply()
    print("nest_asyncio applied -- asyncio.run() works inside Jupyter.")
except ImportError:
    print("nest_asyncio not installed. Install with: pip install nest_asyncio")
    print("Alternatively, use `await` directly in Jupyter 7+ / IPython 7+.")

---
## 1. Native Coroutines and `async`/`await`

A **native coroutine** is defined with `async def`. You drive it with the `await` keyword,
which suspends the coroutine and yields control back to the event loop.

**Guido's trick:** squint and pretend the `async` and `await` keywords are not there.
The remaining code reads like a plain sequential function -- except it magically never blocks.

### Key definitions

| Term | Description |
|---|---|
| **Native coroutine** | Defined with `async def`. Driven by `await`. |
| **Awaitable** | Anything you can `await`: coroutine objects, `asyncio.Task`, objects with `__await__` |
| **Coroutine object** | What you get when you *call* an `async def` function (not yet running) |
| **Event loop** | The scheduler that drives coroutines, calling `.send()` under the hood |

In [ ]:
import asyncio

# A native coroutine: just a function with async def
async def greet(name: str, delay: float) -> str:
    """Simulate an async operation (e.g., network call)."""
    await asyncio.sleep(delay)  # yields control to the event loop
    return f"Hello, {name}!"

# Calling a coroutine function returns a coroutine object -- it does NOT run yet
coro = greet("World", 0.1)
print(f"Type: {type(coro)}")
print(f"Not yet executed -- we need asyncio.run() or await")
coro.close()  # clean up the unawaited coroutine

# Drive it with asyncio.run()
result = asyncio.run(greet("World", 0.1))
print(f"Result: {result}")

In [ ]:
import asyncio

# The all-or-nothing problem: once you go async, every I/O function
# in the call chain must be async too (or delegated to a thread).

async def fetch_data() -> str:
    """Simulates an async network call."""
    await asyncio.sleep(0.1)
    return "data from network"

async def process() -> str:
    """Must be async because it awaits fetch_data."""
    raw = await fetch_data()
    return raw.upper()

async def main() -> None:
    """Must be async because it awaits process."""
    result = await process()
    print(f"Processed: {result}")

# The only non-async entry point:
asyncio.run(main())

---
## 2. The asyncio Event Loop: `run`, `gather`, `as_completed`

The event loop is the heart of asyncio. Key functions:

| Function | Purpose |
|---|---|
| `asyncio.run(coro)` | Start the event loop, run one coroutine, return its result |
| `asyncio.gather(*awaitables)` | Run multiple awaitables concurrently, return results in submission order |
| `asyncio.as_completed(awaitables)` | Yield awaitables as they finish (completion order) |
| `asyncio.create_task(coro)` | Schedule a coroutine without waiting for it |
| `asyncio.to_thread(fn, *args)` | Delegate a blocking function to a thread (Python 3.9+) |

In [ ]:
import asyncio
import time

async def fake_download(name: str, delay: float) -> str:
    """Simulate downloading a resource."""
    await asyncio.sleep(delay)
    return f"{name} ({delay}s)"

async def demo_gather():
    """asyncio.gather: run concurrently, results in submission order."""
    t0 = time.perf_counter()
    results = await asyncio.gather(
        fake_download("alpha", 0.3),
        fake_download("bravo", 0.1),
        fake_download("charlie", 0.2),
    )
    elapsed = time.perf_counter() - t0
    print(f"gather results (submission order): {results}")
    print(f"Total time: {elapsed:.2f}s  (not 0.6s -- they ran concurrently!)")

asyncio.run(demo_gather())

In [ ]:
import asyncio
import time

async def fake_download(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"{name} ({delay}s)"

async def demo_as_completed():
    """asyncio.as_completed: results in COMPLETION order."""
    tasks = [
        fake_download("slow", 0.3),
        fake_download("fast", 0.05),
        fake_download("medium", 0.15),
    ]
    t0 = time.perf_counter()
    for coro in asyncio.as_completed(tasks):
        result = await coro
        elapsed = time.perf_counter() - t0
        print(f"  [{elapsed:.2f}s] {result}")

asyncio.run(demo_as_completed())

In [ ]:
import asyncio
import time

def blocking_io(label: str) -> str:
    """A regular (blocking) function -- e.g., file I/O, legacy library."""
    time.sleep(0.2)  # blocks the calling thread
    return f"{label}: done"

async def demo_to_thread():
    """Delegate blocking work to a thread so the event loop stays responsive."""
    t0 = time.perf_counter()
    # Run three blocking calls concurrently via threads
    results = await asyncio.gather(
        asyncio.to_thread(blocking_io, "file-A"),
        asyncio.to_thread(blocking_io, "file-B"),
        asyncio.to_thread(blocking_io, "file-C"),
    )
    elapsed = time.perf_counter() - t0
    print(f"Results: {results}")
    print(f"Elapsed: {elapsed:.2f}s  (concurrent threads, not 0.6s)")

asyncio.run(demo_to_thread())

---
## 3. Async Iterators and Async Generators

`async for` works with **asynchronous iterables** -- objects implementing:
- `__aiter__()` -- returns the async iterator (regular method, not coroutine)
- `__anext__()` -- coroutine that returns the next value

Just as generator functions simplify the Iterator pattern, **async generator functions**
(`async def` + `yield`) simplify async iterators.

### Key differences from native coroutines

| | Native coroutine | Async generator |
|---|---|---|
| Declaration | `async def` (no `yield`) | `async def` + `yield` |
| Returns | single value via `return` | yields multiple values |
| Driven by | `await` | `async for` |
| Is awaitable? | Yes | No (it is an async iterable) |

In [ ]:
import asyncio

# An async generator function: async def + yield
async def countdown(name: str, n: int):
    """Yield numbers from n down to 1, with async delays."""
    for i in range(n, 0, -1):
        await asyncio.sleep(0.05)  # simulate async I/O between yields
        yield f"{name}: {i}"

async def demo_async_generator():
    # Drive with async for
    async for value in countdown("rocket", 5):
        print(value)

asyncio.run(demo_async_generator())

In [ ]:
import asyncio

async def ticker(label: str, count: int, interval: float):
    """Async generator that yields tick messages."""
    for i in range(count):
        await asyncio.sleep(interval)
        yield f"[{label}] tick {i+1}"

async def multi_ticker():
    """Drive multiple async generators concurrently.

    This mimics the book's multi_probe pattern: create tasks that
    yield results, then process them as they arrive.
    """
    async def collect(gen):
        """Collect all items from an async generator into a list."""
        results = []
        async for item in gen:
            results.append(item)
        return results

    results = await asyncio.gather(
        collect(ticker("A", 3, 0.1)),
        collect(ticker("B", 3, 0.05)),
    )
    for group in results:
        for item in group:
            print(f"  {item}")

asyncio.run(multi_ticker())

In [ ]:
import asyncio

# Async generator expression and async comprehension (PEP 530)
# Must be consumed inside an async function with async for

async def demo_async_genexpr():
    names = ["python.org", "rust-lang.org", "golang.org"]

    # Simulate probing domains
    async def fake_probe(domain):
        await asyncio.sleep(0.01)
        return (domain, len(domain) % 2 == 0)

    # Async generator function (like multi_probe from the book)
    async def multi_probe(domains):
        coros = [fake_probe(d) for d in domains]
        for coro in asyncio.as_completed(coros):
            yield await coro

    # Async list comprehension: built with async for
    results = [result async for result in multi_probe(names)]
    print("Async comprehension results:")
    for domain, found in results:
        mark = "+" if found else " "
        print(f"  {mark} {domain}")

    # Async generator expression (filtering)
    found_gen = (domain async for domain, found in multi_probe(names) if found)
    print("\nFound domains (via async genexpr):")
    async for domain in found_gen:
        print(f"  + {domain}")

asyncio.run(demo_async_genexpr())

---
## 4. Async Context Managers

`async with` works with objects implementing:
- `__aenter__(self)` -- coroutine for setup
- `__aexit__(self, exc_type, exc_val, exc_tb)` -- coroutine for teardown

Common uses: HTTP client sessions, database connections/transactions, semaphores.

You can also use `@asynccontextmanager` from `contextlib` to write async context
managers with an async generator function (just like `@contextmanager` for sync code).

In [ ]:
import asyncio

class AsyncTimer:
    """An async context manager that measures elapsed time."""

    async def __aenter__(self):
        self.start = asyncio.get_event_loop().time()
        print("Timer started.")
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        elapsed = asyncio.get_event_loop().time() - self.start
        print(f"Timer stopped. Elapsed: {elapsed:.3f}s")
        return False  # do not suppress exceptions

async def demo_async_cm():
    async with AsyncTimer() as timer:
        await asyncio.sleep(0.25)
        print("  ... doing async work ...")

asyncio.run(demo_async_cm())

In [ ]:
import asyncio
from contextlib import asynccontextmanager

@asynccontextmanager
async def managed_resource(name: str):
    """Async context manager using @asynccontextmanager.

    Everything before yield -> __aenter__
    Everything after yield  -> __aexit__
    """
    print(f"  Acquiring {name}...")
    await asyncio.sleep(0.05)  # simulate async setup
    try:
        yield f"Resource<{name}>"
    finally:
        print(f"  Releasing {name}...")
        await asyncio.sleep(0.05)  # simulate async teardown

async def demo_contextmanager_decorator():
    async with managed_resource("database") as db:
        print(f"  Using: {db}")
        await asyncio.sleep(0.1)

asyncio.run(demo_contextmanager_decorator())

---
## 5. Semaphores for Throttling Concurrent Requests

`asyncio.Semaphore(n)` limits how many coroutines can hold it concurrently.

- Internal counter starts at `n`.
- `await semaphore.acquire()` -- decrements counter; suspends if counter is 0.
- `semaphore.release()` -- increments counter (not a coroutine).
- Best used as `async with semaphore:` which calls acquire/release automatically.

`BoundedSemaphore` prevents releasing more times than acquiring (raises `ValueError`).

### The flag-download pattern from the book
```python
async def download_one(client, cc, semaphore):
    async with semaphore:          # at most N concurrent downloads
        image = await get_flag(client, cc)
    save_flag(image, f"{cc}.gif")  # release semaphore before saving
```

In [ ]:
import asyncio
import time

async def fetch(url: str, semaphore: asyncio.Semaphore) -> str:
    """Simulate a throttled network request."""
    async with semaphore:
        print(f"  >> Fetching {url} (free slots: {semaphore._value})")
        await asyncio.sleep(0.2)  # simulate network I/O
        print(f"  << Done     {url}")
        return f"{url}: OK"

async def demo_semaphore():
    MAX_CONCURRENT = 3
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)

    urls = [f"page-{i}" for i in range(8)]

    t0 = time.perf_counter()
    results = await asyncio.gather(*(fetch(url, semaphore) for url in urls))
    elapsed = time.perf_counter() - t0

    print(f"\n{len(results)} requests completed in {elapsed:.2f}s")
    print(f"Without throttling: ~0.2s. With {MAX_CONCURRENT} slots: ~{0.2 * (len(urls) / MAX_CONCURRENT):.1f}s")

asyncio.run(demo_semaphore())

In [ ]:
import asyncio

async def demo_bounded_semaphore():
    """BoundedSemaphore raises ValueError on extra release()."""
    sem = asyncio.BoundedSemaphore(2)

    # Normal usage
    await sem.acquire()
    await sem.acquire()
    sem.release()
    sem.release()
    print("Normal acquire/release: OK")

    # Extra release -> ValueError
    await sem.acquire()
    sem.release()
    try:
        sem.release()  # one too many!
    except ValueError as e:
        print(f"BoundedSemaphore caught extra release: {e}")

asyncio.run(demo_bounded_semaphore())

---
## 6. Putting It All Together: Concurrent Downloads with Throttling

This example combines the patterns from the chapter:
- `asyncio.run()` to start the event loop
- `async with` for a session-like context manager
- `asyncio.Semaphore` for throttling
- `asyncio.as_completed()` for processing results as they arrive
- `asyncio.to_thread()` for delegating blocking work

It simulates the book's flag-download examples without needing a network.

In [ ]:
import asyncio
import time
import random
from collections import Counter
from contextlib import asynccontextmanager

# --- Simulated "client session" (like httpx.AsyncClient) ---
@asynccontextmanager
async def async_client():
    print("Opening async client session...")
    await asyncio.sleep(0.01)
    yield "client-session"
    print("Closing async client session.")
    await asyncio.sleep(0.01)

# --- Simulated I/O ---
async def get_flag(client: str, cc: str) -> bytes:
    """Simulate async HTTP GET."""
    delay = random.uniform(0.05, 0.2)
    await asyncio.sleep(delay)
    if random.random() < 0.1:
        raise ConnectionError(f"Timeout fetching {cc}")
    return f"image-{cc}".encode()

def save_flag(data: bytes, filename: str) -> None:
    """Simulate blocking file I/O."""
    time.sleep(0.01)  # blocking!

# --- Download logic ---
async def download_one(client, cc, semaphore, verbose=False):
    try:
        async with semaphore:
            image = await get_flag(client, cc)
        # Delegate blocking save to thread
        await asyncio.to_thread(save_flag, image, f"{cc}.gif")
        if verbose:
            print(f"  {cc}: OK")
        return "ok"
    except ConnectionError as e:
        if verbose:
            print(f"  {cc}: ERROR ({e})")
        return "error"

async def supervisor(cc_list, concur_req=5, verbose=True):
    counter = Counter()
    semaphore = asyncio.Semaphore(concur_req)

    async with async_client() as client:
        tasks = [download_one(client, cc, semaphore, verbose)
                 for cc in cc_list]

        for coro in asyncio.as_completed(tasks):
            status = await coro
            counter[status] += 1

    return counter

# --- Entry point ---
random.seed(42)
countries = ["BR", "CN", "DE", "EG", "FR", "ID", "IN", "JP",
             "MX", "NG", "PH", "PK", "RU", "TR", "US", "VN"]

t0 = time.perf_counter()
counts = asyncio.run(supervisor(countries, concur_req=5))
elapsed = time.perf_counter() - t0

print(f"\n--- Report ---")
for status, count in counts.most_common():
    print(f"  {status:>5}: {count}")
print(f"  Total: {sum(counts.values())} in {elapsed:.2f}s")

---
## 7. How Async Works -- and When It Does Not

### Why async excels at I/O

| Device | CPU cycles | Human scale |
|---|---|---|
| L1 cache | 3 | 3 seconds |
| L2 cache | 14 | 14 seconds |
| RAM | 250 | 250 seconds |
| Disk | 41,000,000 | 1.3 years |
| Network | 240,000,000 | 7.6 years |

Network I/O is **80 million times** slower than L1 cache. While one coroutine waits for
the network (7.6 human-years), the event loop can drive millions of other operations.

### The myth of "I/O-bound systems"

There are no I/O-bound *systems* -- only I/O-bound *functions*. Every nontrivial system
has CPU-bound parts. If any CPU-bound code runs in a coroutine, it **blocks the event loop**
and kills performance for all other coroutines.

### Dealing with CPU-bound code

1. **Delegate to `ProcessPoolExecutor`** via `loop.run_in_executor()`
2. **Use an external task queue** (Celery, RQ, etc.)
3. **Rewrite in C/Cython/Rust** with GIL release
4. **Accept the cost** (document the decision as tech debt)

In [ ]:
import asyncio
import time
import math

# Demonstrate the danger: CPU-bound work blocks the event loop

def cpu_bound_is_prime(n: int) -> bool:
    """CPU-intensive primality test."""
    if n < 2: return False
    if n == 2: return True
    if n % 2 == 0: return False
    for i in range(3, math.isqrt(n) + 1, 2):
        if n % i == 0:
            return False
    return True

async def heartbeat():
    """Prints a tick every 0.1s -- shows whether the event loop is responsive."""
    for i in range(10):
        print(f"  heartbeat {i}", end="  ")
        await asyncio.sleep(0.1)
    print()

async def bad_example():
    """CPU-bound work IN a coroutine -- blocks the event loop!"""
    print("BAD: CPU-bound work directly in coroutine")
    task = asyncio.create_task(heartbeat())
    # This blocks the event loop -- heartbeat won't tick until it's done
    result = cpu_bound_is_prime(5_000_000_029)
    print(f"\n  Prime check result: {result}")
    await task

async def good_example():
    """CPU-bound work delegated to a thread -- event loop stays responsive."""
    print("\nGOOD: CPU-bound work delegated to thread")
    task = asyncio.create_task(heartbeat())
    result = await asyncio.to_thread(cpu_bound_is_prime, 5_000_000_029)
    print(f"\n  Prime check result: {result}")
    await task

asyncio.run(bad_example())
asyncio.run(good_example())

---
## Key Takeaways

1. **`async def` + `await`** define and drive native coroutines. Use Guido's trick:
   ignore the keywords and read the sequential logic.

2. **`asyncio.run()`** is the single entry point from synchronous to asynchronous code.
   Inside async code, use `gather` for parallel work, `as_completed` for progress.

3. **`async for`** drives async iterables (objects with `__aiter__`/`__anext__`).
   Async generators (`async def` + `yield`) are the easiest way to create them.

4. **`async with`** manages async resources (sessions, connections, transactions).
   Use `@asynccontextmanager` for lightweight implementations.

5. **`asyncio.Semaphore`** throttles concurrent access -- essential for polite HTTP clients.
   Use it as `async with semaphore:` and hold it for the shortest possible time.

6. **The all-or-nothing problem:** once you go async, every I/O call in the chain must
   be async or delegated to a thread with `asyncio.to_thread()`.

7. **CPU-bound code is the enemy of async.** There are no I/O-bound *systems*, only
   I/O-bound *functions*. Delegate CPU work to processes, task queues, or native extensions.

---
*Notebook generated from Fluent Python Chapter 21 distillation.*